# Downscaling de LST para Arequipa (U-Net, GOES 2km → VIIRS 1km)

Réplica del enfoque de *Kurchaba & Meyer (2026)* adaptada a Perú: GOES-East en vez de SEVIRI, VIIRS en vez de MODIS, y **DEM como canal extra** para el terreno andino.

**Antes de empezar:**
1. Menú `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`
2. Ten a mano tu usuario/clave de https://urs.earthdata.nasa.gov

El código se clona solo desde GitHub; los datos y checkpoints se guardan en tu Google Drive (carpeta `lst_arequipa`) para sobrevivir a los cortes de sesión de Colab.

In [ ]:
# 1) Preparar entorno: clonar codigo + Drive para datos/checkpoints persistentes
from google.colab import drive
drive.mount('/content/drive')
!git clone -q https://github.com/eduardo020698/lst-arequipa /content/lst-arequipa 2>/dev/null || git -C /content/lst-arequipa pull -q
%cd /content/lst-arequipa
import os
pers = '/content/drive/MyDrive/lst_arequipa'
for d in ['data', 'checkpoints']:
    os.makedirs(f'{pers}/{d}', exist_ok=True)
    if os.path.isdir(d) and not os.path.islink(d):
        os.system(f'rm -rf {d}')
    if not os.path.exists(d):
        os.symlink(f'{pers}/{d}', d)
!pip -q install s3fs earthaccess h5netcdf rasterio 2>/dev/null | tail -1
print('entorno listo')

In [ ]:
# 2) Login en NASA Earthdata (pide usuario y clave de forma interactiva)
import earthaccess
earthaccess.login(strategy='interactive', persist=True)

## Prueba de humo (5 días)
Corre primero esto para validar el pipeline de punta a punta (~10 min). Si funciona, pasa al rango completo.

In [ ]:
!python scripts/download_goes.py --start 2025-06-01 --end 2025-06-05 --hours 6,7,17,18,19
!python scripts/download_viirs.py --start 2025-06-01 --end 2025-06-05
!python scripts/build_dataset.py
!ls data/samples | head

## Dataset completo
1–2 años es un buen punto de partida (la descarga corre sola; puedes cerrar y volver — los scripts saltan lo ya descargado). Las horas 6,7 UTC ≈ pasada nocturna de VIIRS (~01:30 local) y 17,18,19 UTC ≈ pasada diurna (~13:30 local).

In [ ]:
START, END = '2024-07-01', '2026-06-30'   # ajusta el rango aquí
!python scripts/download_goes.py --start {START} --end {END} --hours 6,7,17,18,19
!python scripts/download_viirs.py --start {START} --end {END}
!python scripts/build_dataset.py

In [ ]:
# 3) Entrenar (reanudable: si Colab te corta, vuelve a correr esta celda)
!python train.py --epochs 60 --batch 16 --resume

In [ ]:
# 4) Visualizar un caso de test: GOES crudo vs U-Net vs VIIRS real
import glob, numpy as np, torch, matplotlib.pyplot as plt
import sys; sys.path.insert(0, '.')
from model.unet import UNet
from scripts.build_dataset import LST_MEAN, LST_STD

model = UNet(); model.load_state_dict(torch.load('checkpoints/best.pt', map_location='cpu')); model.eval()
f = sorted(glob.glob('data/samples/sample_*.npz'))[-1]
d = np.load(f); x, y, m = d['x'], d['y'], d['mask']
H8, W8 = (y.shape[0]//8)*8, (y.shape[1]//8)*8
with torch.no_grad():
    p = model(torch.from_numpy(x[None,:,:H8,:W8]))[0,0].numpy()*LST_STD + LST_MEAN
g = x[0,:H8,:W8]*LST_STD + LST_MEAN
v = np.where(m[:H8,:W8], y[:H8,:W8], np.nan)
fig, ax = plt.subplots(1, 3, figsize=(16,5))
for a, img, t in zip(ax, [g, p, v], ['GOES 2 km (entrada)', 'U-Net 1 km (predicho)', 'VIIRS 1 km (real)']):
    im = a.imshow(img-273.15, cmap='inferno', vmin=0, vmax=45); a.set_title(t); a.axis('off')
fig.colorbar(im, ax=ax, label='LST (°C)', shrink=0.8); plt.show()
print('pesos listos en checkpoints/best.pt — traémelos para integrarlos al dashboard')